In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')


In [ ]:
#df = pd.read_csv('steam_indie_merged_data.csv', engine='python')
df = pd.read_csv('../../../data/processed/steam_indie_games.csv')

In [ ]:
df['price'].value_counts().sort_index()

price
0        190
49        78
53         1
54         1
55        19
        ... 
5499      11
5999       2
7999       1
9999       4
19999     15
Name: count, Length: 282, dtype: int64

In [ ]:
df[df['price']==0]

,appid,owners,positive,negative,price,ccu,type,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name
20,2420510,"1,000,000 .. 2,000,000",37623,363,0,645,game,"['Action', 'Adventure', 'Casual', 'Indie']",2023-08-16,KayAnimate,37986,1000000,False,False,HoloCure - Save the Fans!
24,2622000,"1,000,000 .. 2,000,000",9547,7422,0,3227,game,"['Casual', 'Indie', 'RPG', 'Strategy']",2024-02-28,STAR ENGINE PROJECT,16969,1000000,False,False,Astral Party
86,2381590,"500,000 .. 1,000,000",11002,4035,0,22,game,"['Adventure', 'Casual', 'Indie', 'Simulation']",2023-05-24,Indiesolodev,15037,500000,False,False,not available
121,402710,"500,000 .. 1,000,000",8064,6961,0,13,game,"['Action', 'Adventure', 'Indie', 'RPG']",2023-01-18,Fenix Fire Entertainment,15025,500000,False,False,Osiris: New Dawn
146,327070,"200,000 .. 500,000",7014,3324,0,1,game,"['Action', 'Adventure', 'Indie', 'Massively Mu...",2023-02-07,gamigo US,10338,200000,False,False,Gloria Victis: Medieval MMORPG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9073,485770,"0 .. 20,000",10,1,0,0,game,['Indie'],2024-02-02,"mamoniem, Owlnight",11,0,False,False,Penguins of The North
9153,1606350,"0 .. 20,000",27,7,0,0,game,"['Indie', 'Strategy']",2023-05-09,micomuko,34,0,False,False,Chess 432
9231,2319050,"0 .. 20,000",8,21,0,0,game,"['Action', 'Adventure', 'Indie', 'RPG']",2023-04-27,Andy Law,29,0,False,False,闪客快打8武装行
9390,2597670,"0 .. 20,000",11,1,0,0,game,"['Casual', 'Indie', 'Simulation', 'Strategy']",2024-03-31,PrinceXI,12,0,False,False,TD Designer


In [ ]:

print(df.head())
print(df.shape)
print(df.columns.tolist())


# -----------------------------
# 2) 기본 복사
# -----------------------------
df_clean = df.copy()


# -----------------------------
# 3) 문자열 컬럼 정리
# -----------------------------
text_cols = ["name", "owners", "type", "genres", "release_date", "developers"]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("string").str.strip()


# -----------------------------
# 4) owners 분해
# 예: "50,000,000 .. 100,000,000"
# -----------------------------
df_clean["owners"] = df_clean["owners"].str.replace(",", "", regex=False)

df_clean["owners_low"] = df_clean["owners"].str.split(r"\.\.").str[0].str.strip()
df_clean["owners_high"] = df_clean["owners"].str.split(r"\.\.").str[1].str.strip()

df_clean["owners_low"] = pd.to_numeric(df_clean["owners_low"], errors="coerce")
df_clean["owners_high"] = pd.to_numeric(df_clean["owners_high"], errors="coerce")

df_clean["owners_mid"] = (df_clean["owners_low"] + df_clean["owners_high"]) / 2


# -----------------------------
# 5) 수치형 컬럼 변환
# -----------------------------
numeric_cols = ["appid", "positive", "negative", "price", "ccu", "total_reviews", "owners_lower"]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# -----------------------------
# 6) 가격 달러 단위로 변환
# 예: 2999 -> 29.99
# -----------------------------
df_clean["price_usd"] = df_clean["price"] / 100

# -----------------------------
# 7) 리뷰 파생변수
# -----------------------------
df_clean["review_total"] = df_clean["positive"].fillna(0) + df_clean["negative"].fillna(0)

df_clean["positive_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["positive"] / df_clean["review_total"],
    np.nan
)

df_clean["negative_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["negative"] / df_clean["review_total"],
    np.nan
)

df_clean["log_review_total"] = np.log1p(df_clean["review_total"])


# -----------------------------
# 8) genres 문자열 -> 실제 리스트
# -----------------------------
def parse_genres(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

df_clean["genres_list"] = df_clean["genres"].apply(parse_genres)

# -----------------------------
# 9) 장르 플래그 생성
# -----------------------------
def has_genre(genres, genre_name):
    if not isinstance(genres, list):
        return 0
    return int(genre_name in genres)

genre_targets = [
    "Action", "Adventure", "RPG", "Strategy",
    "Simulation", "Casual", "Free To Play", "Early Access", "Indie"
]

for g in genre_targets:
    col_name = "genre_" + g.lower().replace(" ", "_")
    df_clean[col_name] = df_clean["genres_list"].apply(lambda x: has_genre(x, g))


# -----------------------------
# 10) 날짜 처리
# -----------------------------
df_clean["release_date_parsed"] = pd.to_datetime(
    df_clean["release_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

today = pd.Timestamp.today().normalize()

df_clean["days_since_release"] = (today - df_clean["release_date_parsed"]).dt.days
df_clean["release_year"] = df_clean["release_date_parsed"].dt.year
df_clean["release_month"] = df_clean["release_date_parsed"].dt.month

# -----------------------------
# 11) 무료 게임 여부
# -----------------------------
df_clean["is_free"] = (df_clean["price_usd"] == 0).astype(int)



# -----------------------------
# 12) 가격 구간화
# -----------------------------
df_clean["price_band"] = pd.cut(
    df_clean["price_usd"],
    bins=[-1, 0, 5, 10, 20, 30, 60, 9999],
    labels=["Free", "0~5", "5~10", "10~20", "20~30", "30~60", "60+"]
)

# -----------------------------
# 13) 리뷰 규모 구간화
# -----------------------------
df_clean["review_band"] = pd.cut(
    df_clean["review_total"],
    bins=[-1, 10, 50, 100, 500, 1000, 10000, 999999999],
    labels=["0~10", "11~50", "51~100", "101~500", "501~1000", "1001~10000", "10000+"]
)

# -----------------------------
# 14) 이름 정리용 대표 이름 만들기
# store 이름 우선, 없으면 spy 이름 사용
# -----------------------------
df_clean["game_name"] = df_clean["name"]



# -----------------------------
# 15) 최종 확인
# -----------------------------
print(df_clean.head())
print(df_clean.shape)
print(df_clean.columns.tolist())
print(df_clean.isna().sum().sort_values(ascending=False).head(20))

# -----------------------------
# 16) 저장
# -----------------------------
df_clean.to_csv("steam_indie_preprocessed.csv", index=False, encoding="utf-8-sig")
print("저장 완료: steam_indie_preprocessed.csv")

     appid                    owners  positive  negative  price    ccu  type  \
0   899770  20,000,000 .. 50,000,000     88027     22596   3499   5831  game   
1   251570  10,000,000 .. 20,000,000    327889     42157   4499  17045  game   
2  1116170  10,000,000 .. 20,000,000       266        56   1499      3  game   
3  1326470  10,000,000 .. 20,000,000    222495     31051   2999   4450  game   
4  2186680  10,000,000 .. 20,000,000     26360      4445   4999   3582  game   

                                              genres release_date  \
0            ['Action', 'Adventure', 'Indie', 'RPG']   2024-02-21   
1  ['Action', 'Adventure', 'Indie', 'RPG', 'Simul...   2024-07-25   
2            ['Action', 'Adventure', 'Indie', 'RPG']   2025-04-22   
3     ['Action', 'Adventure', 'Indie', 'Simulation']   2024-02-22   
4  ['Action', 'Adventure', 'Indie', 'RPG', 'Strat...   2023-12-07   

            developers  total_reviews  owners_lower  is_f2p  is_early_access  \
0  Eleventh Hour Games  

In [ ]:
print(df_clean[[
    "price",
    "review_total",
    "positive_ratio",
    "owners_mid",
    "days_since_release"
]].describe())

              price   review_total  positive_ratio    owners_mid  \
count   9593.000000    9593.000000     9593.000000  9.593000e+03   
mean     886.018451     801.693214        0.841890  5.922079e+04   
std     1062.199017    7701.831191        0.151725  5.385844e+05   
min        0.000000      10.000000        0.000000  1.000000e+04   
25%      314.000000      19.000000        0.771372  1.000000e+04   
50%      599.000000      41.000000        0.883721  1.000000e+04   
75%     1199.000000     156.000000        0.953125  3.500000e+04   
max    19999.000000  370046.000000        1.000000  3.500000e+07   

       days_since_release  
count         9593.000000  
mean           736.416345  
std            264.296222  
min            124.000000  
25%            524.000000  
50%            727.000000  
75%            955.000000  
max           1215.000000  


In [ ]:
#가격별 리뷰수
print(
    df_clean.groupby("price_band", observed=False)["review_total"]
    .mean()
    .sort_values(ascending=False)
)

price_band
30~60    15439.195122
20~30     3517.108883
10~20     1265.718415
Free       684.342105
5~10       484.767345
0~5        256.594224
60+         20.750000
Name: review_total, dtype: float64


In [ ]:
#장르별 리뷰량
genre_cols = [
    "genre_action", "genre_adventure", "genre_rpg",
    "genre_strategy", "genre_simulation", "genre_casual"
]

for col in genre_cols:
    print(f"\n[{col}]")
    print(df_clean.groupby(col)["review_total"].mean())


[genre_action]
genre_action
0     581.013350
1    1103.724938
Name: review_total, dtype: float64

[genre_adventure]
genre_adventure
0     605.977245
1    1000.493591
Name: review_total, dtype: float64

[genre_rpg]
genre_rpg
0     651.367534
1    1314.085478
Name: review_total, dtype: float64

[genre_strategy]
genre_strategy
0     721.620563
1    1108.201812
Name: review_total, dtype: float64

[genre_simulation]
genre_simulation
0     609.488976
1    1355.368932
Name: review_total, dtype: float64

[genre_casual]
genre_casual
0    1024.402775
1     496.100890
Name: review_total, dtype: float64


In [ ]:
print(df_clean.shape)
print('-'*50)
print(df_clean.columns.tolist())
print('-'*50)
print(df_clean.head())
print('-'*50)
print(df_clean.info())

(9593, 41)
--------------------------------------------------
['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'type', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access', 'name', 'owners_low', 'owners_high', 'owners_mid', 'price_usd', 'review_total', 'positive_ratio', 'negative_ratio', 'log_review_total', 'genres_list', 'genre_action', 'genre_adventure', 'genre_rpg', 'genre_strategy', 'genre_simulation', 'genre_casual', 'genre_free_to_play', 'genre_early_access', 'genre_indie', 'release_date_parsed', 'days_since_release', 'release_year', 'release_month', 'is_free', 'price_band', 'review_band', 'game_name']
--------------------------------------------------
     appid                owners  positive  negative  price    ccu  type  \
0   899770  20000000 .. 50000000     88027     22596   3499   5831  game   
1   251570  10000000 .. 20000000    327889     42157   4499  17045  game   
2  1116170  10000000 .. 20000000       266    

In [ ]:
na_summary = df_clean.isna().sum().sort_values(ascending=False)
print(na_summary.head(20))

developers         12
owners              0
positive            0
negative            0
price               0
ccu                 0
type                0
genres              0
appid               0
release_date        0
total_reviews       0
owners_lower        0
is_f2p              0
is_early_access     0
name                0
owners_low          0
owners_high         0
owners_mid          0
price_usd           0
review_total        0
dtype: int64


In [ ]:
df_clean['positive'].isna().sum()

np.int64(0)

In [ ]:
df_clean['positive_ratio'].isna().sum()

np.int64(0)

In [ ]:
df_clean['negative'].isna().sum()

np.int64(0)

In [ ]:
df_clean['negative_ratio'].isna().sum()

np.int64(0)

In [ ]:
df_clean.sort_values(by='review_total', ascending=False).head()

,appid,owners,positive,negative,price,ccu,type,genres,release_date,developers,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
1,251570,10000000 .. 20000000,327889,42157,4499,17045,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,...,0,1,2024-07-25,644,2024,7,0,30~60,10000+,7 Days to Die
3,1326470,10000000 .. 20000000,222495,31051,2999,4450,game,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,...,0,1,2024-02-22,798,2024,2,0,20~30,10000+,Sons Of The Forest
8,1144200,5000000 .. 10000000,210654,25542,4999,4296,game,"['Action', 'Adventure', 'Indie']",2023-12-13,VOID Interactive,...,0,1,2023-12-13,869,2023,12,0,30~60,10000+,Ready or Not
5,526870,10000000 .. 20000000,225479,6585,3999,12596,game,"['Adventure', 'Indie', 'Simulation', 'Strategy']",2024-09-10,Coffee Stain Studios,...,0,1,2024-09-10,597,2024,9,0,30~60,10000+,Satisfactory
16,1468810,2000000 .. 5000000,121020,106084,1299,4116,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2023-05-26,鬼谷工作室,...,0,1,2023-05-26,1070,2023,5,0,10~20,10000+,鬼谷八荒 Tale of Immortal


In [ ]:
df_clean[df_clean['review_total'] > 30].sort_values(by='review_total', ascending=True).head()

,appid,owners,positive,negative,price,ccu,type,genres,release_date,developers,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
6373,3214390,0 .. 20000,30,1,599,4,game,"['Action', 'Adventure', 'Indie']",2024-11-29,GoodLuck3 Inc.,...,0,1,2024-11-29,517,2024,11,0,5~10,11~50,Stranger Watch
3189,3152660,0 .. 20000,29,2,399,0,game,"['Casual', 'Indie', 'Strategy']",2024-10-24,Solutena Studio,...,0,1,2024-10-24,553,2024,10,0,0~5,11~50,Bloody Rune
7668,1035600,0 .. 20000,13,18,2499,1,game,"['Action', 'Indie', 'Massively Multiplayer', '...",2024-10-14,PHOSPHORUS GAMES,...,0,1,2024-10-14,563,2024,10,0,20~30,11~50,The Dawn: Sniper's Way
7680,2817910,0 .. 20000,27,4,299,0,game,"['Action', 'Indie', 'RPG']",2024-10-23,Astrow Games,...,0,1,2024-10-23,554,2024,10,0,0~5,11~50,The Nightwatch
8967,1592720,0 .. 20000,29,2,999,0,game,"['Action', 'Indie']",2023-12-08,xeetsh,...,0,1,2023-12-08,874,2023,12,0,5~10,11~50,Garbage Crew!


In [ ]:
df_clean["genres_list"].dtype

dtype('O')

In [ ]:
all_genres = df_clean["genres_list"].explode().dropna()

print(all_genres.unique())
print("장르 개수:", len(all_genres.unique()))

<StringArray>
[               'Action',             'Adventure',                 'Indie',
                   'RPG',            'Simulation',              'Strategy',
 'Massively Multiplayer',                'Casual',                'Racing',
                'Sports',             'Utilities',            'Accounting',
      'Video Production', 'Design & Illustration',         'Photo Editing',
             'Education',     'Software Training',  'Animation & Modeling',
      'Game Development',        'Web Publishing',      'Audio Production']
Length: 21, dtype: str
장르 개수: 21


In [ ]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    9588
Adventure                4759
Action                   4050
Casual                   4044
Simulation               2472
RPG                      2176
Strategy                 1987
Sports                    335
Racing                    297
Massively Multiplayer     121
Utilities                  21
Design & Illustration      10
Animation & Modeling        7
Education                   6
Software Training           6
Game Development            5
Video Production            4
Audio Production            4
Photo Editing               2
Web Publishing              2
Accounting                  1
Name: count, dtype: int64

In [ ]:
df_2024_after = df_clean[df_clean["release_date_parsed"] >= "2024-01-01"].copy()
print("-"*50)
print(df_2024_after.shape)
print("-"*50)
print(df_2024_after.head())
print("-"*50)
print(df_2024_after.tail())

--------------------------------------------------
(6143, 41)
--------------------------------------------------
     appid                owners  positive  negative  price    ccu  type  \
0   899770  20000000 .. 50000000     88027     22596   3499   5831  game   
1   251570  10000000 .. 20000000    327889     42157   4499  17045  game   
2  1116170  10000000 .. 20000000       266        56   1499      3  game   
3  1326470  10000000 .. 20000000    222495     31051   2999   4450  game   
5   526870  10000000 .. 20000000    225479      6585   3999  12596  game   

                                              genres release_date  \
0            ['Action', 'Adventure', 'Indie', 'RPG']   2024-02-21   
1  ['Action', 'Adventure', 'Indie', 'RPG', 'Simul...   2024-07-25   
2            ['Action', 'Adventure', 'Indie', 'RPG']   2025-04-22   
3     ['Action', 'Adventure', 'Indie', 'Simulation']   2024-02-22   
5   ['Adventure', 'Indie', 'Simulation', 'Strategy']   2024-09-10   

             de

In [ ]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    9588
Adventure                4759
Action                   4050
Casual                   4044
Simulation               2472
RPG                      2176
Strategy                 1987
Sports                    335
Racing                    297
Massively Multiplayer     121
Utilities                  21
Design & Illustration      10
Animation & Modeling        7
Education                   6
Software Training           6
Game Development            5
Video Production            4
Audio Production            4
Photo Editing               2
Web Publishing              2
Accounting                  1
Name: count, dtype: int64

In [ ]:
df.to_csv("preprocesing(1).csv")